# 8. 准生产级灰箱系统实操

本 Notebook 运行 `industrial_greybox/` 包的端到端流程，重点不是追求最低误差，而是观察**工程行为**：

- 数值积分在准刚性工况下是否稳定；
- 化验死时间对齐是否正确（残差不应有滞后结构）；
- 清洗规则是否只标记、不伪造；
- 慢变参数修正是否在训练起点等于机理模型；
- 影子运行的日志是否只使用当时已到达的信息。

**边界：** 全部数据由 `synthetic_plant` 生成，不代表任何真实装置验证。

## 8.1 环境与导入

Keras 后端必须在导入 `keras` 前设置。若你的环境没有安装 TensorFlow，请先按 `tutorials/index.md` 创建环境。

In [ ]:
import os

# 后端和 TensorFlow 日志级别都在导入相关库前设置；setdefault 不覆盖用户已有选择。
os.environ.setdefault("KERAS_BACKEND", "tensorflow")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")

import numpy as np
import pandas as pd

from industrial_greybox.config import (
    COL_C_A,
    COL_RUN_ID,
    COL_TIME_MIN,
    DelayConfig,
    ModelConfig,
    ShadowConfig,
)
from industrial_greybox.delays import ShadowBuffer
from industrial_greybox.diagnostics import (
    parameter_identifiability,
    residual_structure,
    sensitivity_correlation_matrix,
    sensitivity_matrix,
)
from industrial_greybox.numerics import explicit_stability_ratio
from industrial_greybox.pipeline import run_pipeline
from industrial_greybox.synthetic_plant import CampaignSpec, simulate_campaign

pd.set_option("display.width", 120)

## 8.2 数值稳定性：先算稳定比，再选积分器

显式 Euler 的稳定条件约为 `h·(1/τ + UA) < 2`。下面这条准刚性工况已经越界，
因此生产默认使用 RK4（可加子步），并保留 IMEX 作为线性刚性更强的选项。

In [ ]:
# 每个元组依次是：工况名、步长 h、停留时间 tau、冷却系数 UA。
# 稳定比 h*(1/tau+UA)：三例分别约 0.28、2.96、3.46；后两例越过 Euler 的约 2 上限。
cases = [
    ("常规工况", 1.0, 5.0, 0.08),
    ("短停留时间", 1.0, 0.35, 0.10),
    ("短停留 + 强冷却", 1.0, 0.35, 0.60),
]
rows = []
for label, dt, tau, ua in cases:
    # 把理论稳定性判断变成可读报表；比值越大，显式步进越容易振荡/发散。
    ratio = explicit_stability_ratio(dt, tau, ua)
    rows.append(
        {
            "工况": label,
            "dt (min)": dt,
            "tau (min)": tau,
            "UA (min^-1)": ua,
            "h*a_max": round(ratio, 3),
            "显式 Euler": "稳定" if ratio < 1 else ("边缘" if ratio < 2 else "可能发散"),
        }
    )
pd.DataFrame(rows)

## 8.3 生成带漂移、时滞和仪表故障的 campaign

隐藏真值与名义机理不同：真活化能 48 kJ/mol（名义 42）、结垢使 UA 随时间衰减、
催化剂活性随累计进料下降。化验结果带 12 min 报告延迟，时间戳表示**结果到达**时刻。

In [ ]:
# 10 条运行各 200 步；生成器同时构造高频 DCS 和较低频的实验室化验记录。
dcs, lab = simulate_campaign(
    CampaignSpec(n_runs=10, n_steps=200, days_span=30, lab_interval_min=10)
)
print(f"DCS 行数 {len(dcs)}，运行 {dcs[COL_RUN_ID].nunique()} 条；化验 {len(lab)} 条")
print("时间戳语义：结果到达时刻（需减死时间才能对齐到过程时刻）")
dcs[
    [
        COL_RUN_ID,
        COL_TIME_MIN,
        "c_a_online_mol_L",  # 在线分析仪（高频、有噪声）
        "c_a_truth_mol_L",   # 仅合成数据才有；现场数据无此列
        "reactor_temperature_K",
    ]
].head()

## 8.4 端到端流程：清洗 → 对齐 → 窗口 → 训练 → 评估 → 影子运行

为便于在 Notebook 中快速迭代，这里只用较少的 epoch 和较大的窗口步长。
真实训练应使用更多的运行数与 `stride=1~2`。

In [ ]:
# 总分析时延由管线 2 min + 分析仪/报告 10 min 组成；匹配化验时必须回到过程时刻。
delay = DelayConfig(line_delay_min=2.0, analysis_delay_min=10.0, lab_interval_min=10.0)

# 一次调用串起清洗、时延对齐、窗口化、训练和评估，result 同时保存中间诊断供复核。
result = run_pipeline(
    dcs,
    lab,
    config=ModelConfig(),
    delay=delay,
    shadow=ShadowConfig(),
    epochs=8,
    stride=8,
    run_baseline=True,
    run_shadow=True,
    verbose=0,
)

print(result.hygiene_report.summary_line())
print()
print("[测试指标：开环窗口，按通道各自归一化后比较]")
result.metrics

## 8.5 清洗报告：可疑值、插值与信息性标记要分开看

“死区”标记属于信息性标记（真实稳态也会触发），不计入可疑值；
被规则删除的值**不会**再被插值“复活”。

In [ ]:
# 把数据质量统计转成表格，先核对被标记/删除/插值的数量，再解读模型误差。
report_frame = result.hygiene_report.to_frame()
report_frame

In [ ]:
# 抽查：被硬边界删除的点应保持 NaN，且 flag 中不含 imputed。
# 这是“标记≠删除”与“删除后不插值”两条原则的直接验证。
cleaned = result.cleaned
flagged = cleaned["c_a_online_mol_L__flag"].fillna("")
limit_rows = cleaned[flagged.str.contains("limit", regex=False)]

if limit_rows.empty:
    print("本批数据没有命中硬边界规则；可调大故障注入比例再看。")
else:
    display(limit_rows[["c_a_online_mol_L", "c_a_online_mol_L__flag"]].head(8))
    print("其中仍为 NaN 的比例：", round(limit_rows["c_a_online_mol_L"].isna().mean(), 3))
    print(
        "被标记为 limit 的点中，误插值的数量：",
        int(limit_rows["c_a_online_mol_L__flag"].str.contains("imputed").sum()),
    )

## 8.6 时滞对齐检查

化验事件日志中 `result_time - process_time` 必须恒等于总死时间。
若残差与过程时刻存在明显斜率，通常说明死时间估计有偏差或对齐方向反了。

In [ ]:
# 事件表保留结果时刻、过程时刻和死时间，可检查对齐计算本身是否正确。
events = result.shadow_lab_events
display(events.head())

# 每条化验的 (result_time - process_time) 必须恒等于总死时间；
# 否则说明对齐方向反了或死时间用错了。
print(
    "死时间一致性：",
    np.allclose(
        events["result_time_min"] - events["process_time_min"],
        events["dead_time_min"],
    ),
)
print("历史不足（缓冲区太短）的事件数：", int(result.shadow_report.n_stale_history))

residual_structure(
    events["residual_grey"].to_numpy(),
    time_min=events["process_time_min"].to_numpy(),
)

### 影子缓冲区为什么必须按时间戳查询

下面用 `ShadowBuffer` 直接演示：化验在 `result_time` 到达，但对应的是
`process_time` 的预测值。若用当前值去比，会把死时间内的真实变化算成残差，
而这部分变化并不是模型误差。

In [ ]:
# 缓冲最近 60 min 的预测；zoh 是零阶保持，即查询时使用最近一个已存预测。
buffer = ShadowBuffer(horizon_min=60.0, interpolation="zoh")
for t in range(0, 41):
    buffer.push(t, [1.0 + 0.01 * t])  # 每分钟增加 0.01 mol/L，例如 t=23 时预测 1.23

result_time, dead_time = 35.0, 12.0
# 35 min 到达的化验若有 12 min 死时间，代表 23 min 的过程状态。
process_time = result_time - dead_time

at_result = float(buffer.query(result_time)[0])
at_process = float(buffer.query(process_time)[0])
print("当前时刻预测（错误做法）:", at_result)
print("过程时刻预测（正确做法）:", at_process)
# 本例当前预测 1.35、对应过程预测 1.23，错用当前值将多出 0.12 mol/L 的假残差。
print("两者差值即会被误算进残差:", at_result - at_process)
print("超出缓冲时长时返回:", buffer.query(-100.0))

## 8.7 影子运行日志与滚动报表

`mechanistic_*` 与 `greybox_*` 共用同一个积分器，因此 `correction_*`
纯粹反映网络修正，不混入数值格式差异。

In [ ]:
log = result.shadow_log
columns = [
    "time_min",
    "c_a0_mol_L",
    "reactor_temperature_K",
    "mechanistic_c_a",
    "greybox_c_a",
    "correction_c_a",
    "lab_c_a",
    "residual_vs_lab",
    "in_domain",
]
display(log[columns].head(12))
print(result.shadow_report.to_text())

## 8.8 参数可辨识性：拟合好 ≠ 参数有物理意义

稳态数据上，`UA` 增大与反应热减小可以给出相同的温度。这里用有限差分
灵敏度计算条件数与参数相关性，让病态立刻可见。条件数越大，说明至少
一个参数的变化方向几乎可以被另一个参数抵消。

In [ ]:
test_set = result.datasets["test"]

# 分别分析浓度与温度通道；max_windows 限制计算量，不改变被选窗口的定义。
for channel, label in [(0, "C_A"), (1, "T")]:
    sens = sensitivity_matrix(
        result.training.model, test_set, channel=channel, max_windows=16
    )
    summary = parameter_identifiability(sens)
    print(f"{label} 条件数 = {summary['condition_number']:.3f}")
    display(sensitivity_correlation_matrix(sens).round(3))

## 8.9 学习到的参数与名义值

在合成数据中真值是 `k_ref ≈ 0.032 min⁻¹`（参考温度 350 K）、`UA ≈ 0.10 min⁻¹`
（清洗后），但训练数据只覆盖 30 天，因此学到的是窗口内的**平均**值。
不要用“参数是否等于真值”作为唯一判据；更重要的是预测误差与残差结构。

In [ ]:
pd.DataFrame([
    {"来源": "灰箱学习值", **result.training.model.fitted_parameters()},
    {"来源": "机理基线学习值", **(result.baseline.model.fitted_parameters() if result.baseline else {})},
    {"来源": "合成真值（参考）", "k_ref_per_min": 0.032, "ua_per_min": 0.10},
])

## 8.8 自适应重锚：区分“仪表坏值”与“真实漂移”

固定限值 `clip(residual, ±max)` 有两个毛病：无法区分坏值与漂移，且瞬时
修正会让状态阶跃。`AdaptiveReanchor` 用稳健 MAD 尺度 + 连续确认 + 指数
平滑解决这两点。

下面直接演示三种情形：正常残差、孤立坏值、持续漂移。

In [ ]:
from dataclasses import replace

from industrial_greybox.config import ShadowConfig
from industrial_greybox.reanchor import AdaptiveReanchor

# 将正常小误差、一次离群毛刺、连续漂移分开喂给同一个更新接口，比较门控行为。
rows = []

# 情形一：稳定的小残差 —— 应被正常修正（约 α=0.5 倍）。
normal = AdaptiveReanchor(ShadowConfig())
for value in (0.005, -0.006, 0.004, 0.005):
    decision = normal.update(value)
    rows.append(("正常残差", value, decision.confidence,
                 decision.applied_bias_mol_l, decision.drift_confirmed))

# 情形二：孤立坏值 —— 应被门控拒绝（置信度 0，不施加）。
spike = AdaptiveReanchor(ShadowConfig())
for value in (0.005, -0.006, 0.004):
    spike.update(value)
# 0.5 mol/L 与前面的毫摩尔级残差相差很大，用来模拟一次孤立坏化验。
decision = spike.update(0.5)
rows.append(("孤立坏值", 0.5, decision.confidence,
             decision.applied_bias_mol_l, decision.drift_confirmed))

# 情形三：持续大偏差 —— 连续确认后应恢复置信度并施加修正。
drift = AdaptiveReanchor(ShadowConfig(drift_confirm_count=3))
# 连续多次出现同向大残差才视作漂移；第 4 次可展示确认后的状态。
for _ in range(4):
    decision = drift.update(0.5)
rows.append(("持续漂移(第4次)", 0.5, decision.confidence,
             decision.applied_bias_mol_l, decision.drift_confirmed))

pd.DataFrame(
    rows, columns=["情形", "残差(mol/L)", "置信度", "施加量(mol/L)", "判定为漂移"]
)

## 8.9 域外渐近门控：修正网络什么时候应该“闭嘴”

修正网络只在训练域内有数据支持。`domain_gate` 用归一化特征的盒距离判断
离训练域有多远，在 `box` 与 `box+taper` 之间平滑过渡。

**关键物理判断**：两类修正的域外行为**必须不同**。

- 符号自由的速率修正、慢变**残余**项 → 回退到 0；
- 单调的结垢/老化**主趋势** → **保持**（老装置本来就该结垢更多）。

若主趋势也回退到 0，等于假设老装置换热面自动变干净，会高估 UA、低估
出口浓度——方向上危险。

In [ ]:
# 直接查看训练模型中的物理层对不同状态/操作量给出的域内外判定。
layer = result.training.model.physics_layer

inside_state = np.array([[1.0, 350.0]], dtype="float32")
inside_control = np.array([[1.1, 350.0, 300.0, 5.0]], dtype="float32")
normal_slow = np.array([[0.5, 0.5, 0.0]], dtype="float32")

cases = {
    "域内（典型工况）": (inside_control, normal_slow),
    "温度高 40 K（外推）": (
        np.array([[1.1, 390.0, 340.0, 5.0]], dtype="float32"),
        normal_slow,
    ),
    "tau=60 min（外推）": (
        np.array([[1.1, 350.0, 300.0, 60.0]], dtype="float32"), normal_slow
    ),
    "老装置（9 倍参考寿命）": (
        inside_control, np.array([[0.9, 0.9, 0.0]], dtype="float32")
    ),
}

pd.DataFrame(
    [
        {"工况": label, **layer.domain_status(inside_state, control, slow)}
        for label, (control, slow) in cases.items()
    ]
)

## 8.10 在线微调与安全熔断（概念演示）

离线模型跑几个月后会遇到概念漂移。`OnlineAdapter` 的策略是：只微调修正
层、**冻结白箱机理核心**，并用独立留出的数据做熔断，指标变差就回滚。

这里不实际训练（耗时），只演示触发判据与策略校验：

In [ ]:
from industrial_greybox.adaptation import AdaptationPolicy, OnlineAdapter

# 策略阈值必须来自工艺/质量部门的可接受风险，不是从示例臆造。
# 以下数值是演示阈值而非推荐工艺限值；真实阈值必须由风险/质量要求确定。
policy = AdaptationPolicy(
    trigger_rmse_mol_l=0.02,     # 滚动 RMSE 超过 0.02 mol/L 视为漂移
    trigger_patience=2,          # 连续两个评估周期超限才触发
    learning_rate=5e-5,          # 比离线低两个数量级
    max_rmse_regression=0.02,    # guard 指标恶化超过 2% 就回滚
)

# 连续两次超出 0.02 才触发：0.005 不超限，0.030 是首次超限，0.031 才满足连续条件。
adapter = OnlineAdapter(result.training.model, policy)
print("观察 0.005 mol/L（正常）  -> 触发:", adapter.observe(0.005))
print("观察 0.030 mol/L（超限）  -> 触发:", adapter.observe(0.030))
print("观察 0.031 mol/L（连续）  -> 触发:", adapter.observe(0.031))
print()
print("机理参数数量（必须冻结）:", len(__import__(
    "industrial_greybox.adaptation", fromlist=["physics_weights"]
).physics_weights(result.training.model)))

## 讨论与练习

1. 把 `IntegratorConfig.method` 改成 `"euler"`，在 `τ=0.35 min` 的工况上
   观察温度轨迹。用 8.2 的稳定比解释现象。
2. 把 `analysis_delay_min` 设为 0（但仍按“结果到达”解释时间戳），
   观察 8.6 的残差斜率。为什么这不属于模型误差？
3. 关闭毛刺规则后重训。误差下降是否说明规则“有害”？还是说明模型在
   拟合坏值？
4. 把 `SlowDriftConfig.initial_decay_bias` 从 -6 改为 0，观察开环偏差
   `bias` 的变化。解释为什么初始偏置会改变系统偏差。
5. 尝试用 `Loss.call` 实现逐点掩码，确认它不生效；改用
   `Model.compute_loss`。总结这对工业标签设计的影响。

**部署前提醒：** 本 Notebook 的所有阈值都来自教学假设。接入现场前，
必须由工艺、仪表和安全人员逐项审核，并用独立批次/时间段留出的数据验证。